In [2]:
pip install --upgrade azure-search-documents azure-core

  Attempting uninstall: azure-core
    Found existing installation: azure-core 1.38.2
    Uninstalling azure-core-1.38.2:
      Successfully uninstalled azure-core-1.38.2
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ============================
# 03_build_index.ipynb
# Azure AI Search 인덱스 생성 + 스키마 설정
# ============================

import os
from dotenv import load_dotenv
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, 
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    VectorSearchVectorizer, AzureOpenAIVectorizer, SemanticConfiguration,
    SemanticPrioritizedFields, SemanticField, SemanticSearch
)
from azure.core.credentials import AzureKeyCredential

# 1. .env 로드
load_dotenv()

endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
key = os.getenv("AZURE_SEARCH_API_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME", "movies-index")

credential = AzureKeyCredential(key)
client = SearchIndexClient(endpoint=endpoint, credential=credential)

# 2. 인덱스 스키마 정의 (Deping 설계서 그대로)
fields = [
    SimpleField(name="id", type="Edm.String", key=True, sortable=True),
    SearchableField(name="title", type="Edm.String", analyzer_name="ko.lucene"),
    SearchableField(name="title_ko", type="Edm.String", analyzer_name="ko.lucene"),
    SimpleField(name="year", type="Edm.Int32", filterable=True, sortable=True),
    SimpleField(name="runtime", type="Edm.Int32", filterable=True, sortable=True),
    SimpleField(name="rating_imdb", type="Edm.Double", filterable=True, sortable=True),
    SearchableField(name="overview", type="Edm.String", analyzer_name="ko.lucene"),
    SearchField(
        name="overview_vector",
        type="Collection(Edm.Single)",
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="movie-vector-profile"
    ),
    SearchableField(name="director", type="Edm.String", filterable=True),
    SimpleField(name="tmdb_id", type="Edm.Int32"),
    # 인지 속성 필드
    SimpleField(name="pacing", type="Edm.String", filterable=True),           # fast / medium / slow
    SimpleField(name="plot_complexity", type="Edm.Int32", filterable=True),   # 1~5
    SimpleField(name="emotional_intensity", type="Edm.Int32", filterable=True), # 1~5
    SimpleField(name="visual_score", type="Edm.Double", filterable=True),
    SimpleField(name="sentiment_score", type="Edm.Double", filterable=True),
    # 컬렉션 필드
    SearchableField(name="genres", type="Collection(Edm.String)", filterable=True, facetable=True),
]

# 3. Vector Search 설정 (HNSW + OpenAI embedding)
vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
    profiles=[
        VectorSearchProfile(
            name="movie-vector-profile",
            algorithm_configuration_name="hnsw-config",
            vectorizer_name="openai-vectorizer"
        )
    ],
    vectorizers=[
        AzureOpenAIVectorizer(
            name="openai-vectorizer",
            azure_open_ai_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_open_ai_key=os.getenv("AZURE_OPENAI_API_KEY"),
            azure_open_ai_deployment_name=os.getenv("AZURE_OPENAI_DEPLOY_EMBED", "text-embedding-3-small")
        )
    ]
)

# 4. Semantic Search 설정
semantic_config = SemanticConfiguration(
    name="movie-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title_ko"),
        content_fields=[SemanticField(field_name="overview")],
        keywords_fields=[SemanticField(field_name="genres"), SemanticField(field_name="director")]
    )
)
semantic_search = SemanticSearch(configurations=[semantic_config])

# 5. 인덱스 생성
index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

# 6. 인덱스 생성 또는 업데이트
try:
    client.create_or_update_index(index)
    print(f"✅ 성공! 인덱스 '{index_name}'가 생성되었습니다.")
except Exception as e:
    print(f"❌ 오류: {e}")

TypeError: key must be a string.